<a href="https://colab.research.google.com/github/DA-CristianGutierrez/Proyecto-final-TT---Rappiplus/blob/main/S12%20Estudiante_Proyecto_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python)

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Explorar la estructura de los datasets del negocio antes de analizarlos.

---

In [74]:
# importar librerías
import pandas as pd

In [75]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog =  pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [76]:
# explorar datasets
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


In [77]:
# Porcentaje de nulls en columna con máximo faltante
300 * 100 / 24800

1.2096774193548387

In [78]:
catalog.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 356.0+ bytes


In [79]:
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


In [80]:
# Porcentaje de nulls columna canal
101 * 100 / 1620

6.234567901234568

**Diagnóstico de nulls:**

- **Tabla orders:** Respecto a este dataset se detectaron valores NULL en varias columnas categóricas como país, dispoitivo, nombre producto, categoría producto. Esto puede impactar en el analisis al realizar segmentaciones, sin embargo, dado que no son cantidades significativas de faltantes de acuerdo con estimaciones de analisis sugeridas (mas del 3%) podemos excluir estos faltantes del analisis. Respecto a las columnas numéricas con nulls procederemos de la misma manera ya que sus faltantes son todavía menos numerosos.
- **Tabla catalog:** Datos completos.
- **Tabla marketing:** La columna canal contiene un porcentaje significativo de nulls lo cual puede distorsionar analisis de inversión de producto al segmentar. Se debe determinar la relevancia del impacto y si los canales tienen relación directa con la columna id_camapaña para contemplar imputar esos faltantes.

---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas

---

In [81]:
# Convertir columnas de fecha a formato fecha
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'])
marketing['fecha'] = pd.to_datetime(marketing['fecha'])

In [82]:
# Verificar valores inválidos en variables numéricas en orders
variables_numericas = orders[['cantidad','precio_unitario','monto_descuento','monto_total']]

variables_numericas.describe()

,cantidad,precio_unitario,monto_descuento,monto_total
count,25050.000000,25050.000000,25050.000000,2.510000e+04
mean,7.092735,259.305549,4.500798,2.072680e+03
std,296.277003,138.726461,5.223010,9.894995e+04
min,-2.000000,20.030000,0.000000,-4.926500e+02
25%,1.000000,138.377500,0.000000,1.805075e+02
50%,2.000000,258.715000,0.000000,3.417500e+02
75%,2.000000,380.332500,10.000000,5.185800e+02
max,20000.000000,499.960000,15.000000,8.840200e+06


In [83]:
# Verificar valores inválidos en variables numéricas en catalog
catalog['costo_unitario'].describe()

,costo_unitario
count,7.000000
mean,102.252857
std,111.011563
min,10.120000
25%,16.905000
50%,25.210000
75%,182.975000
max,280.680000


In [84]:
# Verificar valores inválidos en variables numéricas en marketing
marketing['gasto'].describe()

,gasto
count,1620.00000
mean,1772.74292
std,734.43294
min,501.11000
25%,1128.03000
50%,1782.42500
75%,2420.68500
max,2999.36000


In [85]:
# Eliminar duplicados
orders.drop_duplicates(subset='id_pedido', inplace=True)
catalog.drop_duplicates(subset='nombre_producto', inplace=True)

In [120]:
# Verificamos nuevo tamaño de ambos datasets
print('orders:',len(orders))
print('catalog:',len(catalog))

orders: 25000
catalog: 7


**NOTA:** no se eliminaron duplicados del dataset `marketing`, ya que su identificador único tiene asociado varios gastos aplicados en diferentes fechas. Es decir, cada registro es un gasto aplicado a cada campaña en cuestión.

In [86]:
# Revisión de variables categóricas en orders
variables_categoricas_orders = orders[['pais',
                                'dispositivo',
                                'fuente_referencia',
                                'nombre_producto',
                                'categoria_producto']]
for col in variables_categoricas_orders:
    print(col,':')
    print(variables_categoricas_orders[col].value_counts(normalize=True) * 100)
    print('') # salto de linea por cada columna

pais :
pais
Colombia     30.287449
Mexico       30.275304
Argentina    29.388664
mexico        3.493927
colombia      3.331984
argentina     3.222672
Name: proportion, dtype: float64

dispositivo :
dispositivo
desktop    50.884708
mobile     49.115292
Name: proportion, dtype: float64

fuente_referencia :
fuente_referencia
social         33.648378
organic        33.191830
paid_search    33.159792
Name: proportion, dtype: float64

nombre_producto :
nombre_producto
Blender-XL-Red          16.736083
Jacket-Winter-M         16.724069
Vacuum-Pro-Black        16.724069
Sneakers-Urban-42       16.611934
Laptop-Gaming-16GB      11.141370
Tablet-Standard-64GB    11.089307
Phone-Pro-128GB         10.973168
Name: proportion, dtype: float64

categoria_producto :
categoria_producto
Hogar          33.491172
Moda           33.286517
Electronica    33.222311
Name: proportion, dtype: float64



In [87]:
# formateamos valores que tienen caracteristicas diferentes pero se refieren a la misma categoría
orders['pais'] = orders['pais'].str.lower()
orders['pais'].value_counts(normalize=True) * 100

,proportion
pais,
mexico,33.769231
colombia,33.619433
argentina,32.611336


In [88]:
# Revisión de variables categóricas en catalog
variables_categoricas_cat = catalog[['nombre_producto','categoria_producto','proveedor']]

for col in variables_categoricas_cat:
    print(variables_categoricas_cat[col].value_counts())
    print('') # salto de linea por cada columna

nombre_producto
Laptop-Gaming-16GB      1
Phone-Pro-128GB         1
Tablet-Standard-64GB    1
Blender-XL-Red          1
Vacuum-Pro-Black        1
Sneakers-Urban-42       1
Jacket-Winter-M         1
Name: count, dtype: int64

categoria_producto
Electrónica    3
Hogar          2
Moda           2
Name: count, dtype: int64

proveedor
Fuller, Pena and Myers     1
King Ltd                   1
Bowers LLC                 1
Long-Reid                  1
Rivera, Carr and Finley    1
Greene-Smith               1
Mcmillan-Rhodes            1
Name: count, dtype: int64



---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [89]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False,  encoding='utf-8-sig')
catalog.to_csv('catalog_clean.csv', index=False,  encoding='utf-8-sig')
marketing.to_csv('marketing_clean.csv', index=False,  encoding='utf-8-sig')
# Agregué el argumento 'encoding='utf-8-sig' para remplazar los carácteres que remplazan las tildes y poder crear las visualizaciones en tableu

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)?
- ¿Cuál es el costo total?
- ¿Cuánto se ha invertido en marketing?
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden?
- ¿Cuál es la cantidad promedio de productos por orden?
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal?

In [90]:
# Unión de tablas para calculos
merged_orders_cat = pd.merge(orders, catalog, on=['nombre_producto'], how='left')

<class 'pandas.core.frame.DataFrame'>
Index: 25000 entries, 0 to 24999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           25000 non-null  object        
 1   id_usuario          25000 non-null  object        
 2   fecha_hora_pedido   25000 non-null  datetime64[ns]
 3   pais                24700 non-null  object        
 4   dispositivo         24980 non-null  object        
 5   fuente_referencia   24970 non-null  object        
 6   nombre_producto     24970 non-null  object        
 7   categoria_producto  24920 non-null  object        
 8   cantidad            24950 non-null  float64       
 9   precio_unitario     24950 non-null  float64       
 10  monto_descuento     24950 non-null  float64       
 11  monto_total         25000 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 2.5+ MB


In [112]:
# Confirmamos cantidad de registros como previsión
len(merged_orders_cat)

25000

In [121]:
# Rentabilidad del negocio: ingreso, costos, inversión, rentabilidad
print(f"Ingreso total: {orders['monto_total'].sum():.2f}")
print(f"Costo total: {(merged_orders_cat['costo_unitario'] * merged_orders_cat['cantidad']).sum():.2f}")
print(f"Gasto total: {marketing['gasto'].sum():.2f}")
print(f"Profit: {(orders['monto_total'].sum()
 - (merged_orders_cat['costo_unitario'] * merged_orders_cat['cantidad']).sum()
 - (marketing['gasto']).sum()):.2f}")

Ingreso total: 51987462.26
Costo total: 43124018.41
Gasto total: 2871843.53
Profit: 5991600.32


In [92]:
# Comportamiento de ventas
print(f"Ticket promedio: {orders['monto_total'].mean():.2f}")
print(f"Promedio de productos por orden: {orders['cantidad'].mean():.2f}")
print(f"Producto más vendido: {orders['nombre_producto'].value_counts().index[0]}")

Ticket promedio: 2079.50
Promedio de productos por orden: 7.12
Producto más vendido: Blender-XL-Red


In [93]:
# Total de gasto por canal
marketing.groupby('canal')['gasto'].sum()

,gasto
canal,
organic,913533.01
paid_search,863088.21
social,918043.21


---


## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [94]:
# instalar driver para crear conexión con SQL
!pip install psycopg

In [95]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [96]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [97]:
# PARTE 1: Totales del funnel
# ======================

# Explorar etapas del funnel y revisión de consistencia
query_totals = '''
SELECT DISTINCT nombre_evento,
COUNT(DISTINCT id_usuario) AS total_procesos
FROM events
GROUP BY nombre_evento
ORDER BY total_procesos DESC;
'''


totals = pd.read_sql(query_totals, con=engine)
totals


,nombre_evento,total_procesos
0,first_visit,7796
1,add_to_cart,7634
2,select_item,7582
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [98]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH cte_first_visit AS(
SELECT DISTINCT id_usuario
FROM events
WHERE nombre_evento	= 'first_visit'
),

cte_item AS(
SELECT DISTINCT id_usuario
FROM events
WHERE nombre_evento	= 'select_item'
),

cte_add_to_cart AS(

SELECT DISTINCT id_usuario
FROM events
WHERE nombre_evento	= 'add_to_cart'
),

cte_begin_checkout AS(
SELECT DISTINCT id_usuario
FROM events
WHERE nombre_evento	= 'begin_checkout'
),

cte_purchase AS(
SELECT DISTINCT id_usuario
FROM events
WHERE nombre_evento	= 'purchase'
),

cte_add_payment AS(
SELECT DISTINCT id_usuario
FROM events
WHERE nombre_evento	= 'add_payment_info'
)

SELECT
ROUND((SELECT COUNT (*) FROM cte_item) * 100 / NULLIF((SELECT COUNT (*) FROM cte_first_visit), 0), 2) AS conversión_item,
ROUND((SELECT COUNT (*) FROM cte_add_to_cart) * 100 / NULLIF((SELECT COUNT (*) FROM cte_first_visit), 0), 2) AS conversión_add_cart,
ROUND((SELECT COUNT (*) FROM cte_begin_checkout) * 100 / NULLIF((SELECT COUNT (*) FROM cte_first_visit), 0), 2) AS conversión_checkout,
ROUND((SELECT COUNT (*) FROM cte_purchase) * 100 / NULLIF((SELECT COUNT (*) FROM cte_first_visit), 0), 2) AS conversión_purchase,
ROUND((SELECT COUNT (*) FROM cte_add_payment) * 100 / NULLIF((SELECT COUNT (*) FROM cte_first_visit), 0), 2) AS conversion_add_payment;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,conversión_item,conversión_add_cart,conversión_checkout,conversión_purchase,conversion_add_payment
0,97.0,97.0,92.0,80.0,80.0


- `conversión_add_payment` Es el proceso en el cual se presenta más drop-off de usuarios.
- Hay una eficiencia del 80% de conversión (última etapa del funnel).

<div class="alert alert-block alert-warning">
<b>Comentario del revisor. (Iteración 1)</b> <a class="tocSkip"></a><br><br>

Cristian, es positivo que hayas utilizado <code>groupby('canal')['gasto'].sum()</code> para identificar cómo se distribuye la inversión de marketing entre los distintos canales.

Esta agrupación responde directamente a una de las preguntas de negocio del proyecto.

Quiero señalarte una consideración relacionada con el diagnóstico de valores faltantes que realizaste anteriormente.

En el dataset de marketing identificaste 101 registros sin información en la columna <code>canal</code>.

Cuando realizas un <code>groupby()</code> sobre esta variable, pandas excluye de forma predeterminada esos registros de las agrupaciones.

Por esta razón, la suma del gasto agrupado por canal no necesariamente coincide con el gasto total de marketing.

<b>Recomendación:</b> Revisa si puedes recuperar el canal a partir de <code>id_campaña</code>, como tú mismo propusiste en tu diagnóstico inicial. Si no es posible, puedes conservar temporalmente esos gastos en una categoría identificada como canal desconocido.

Esto permitirá que el análisis por canal represente la inversión total y que no se pierda información relevante.

Muy bien por haber detectado este punto desde la exploración; ahora puedes aprovechar ese hallazgo para fortalecer las agrupaciones.

</div>

<div class="alert alert-block alert-warning">
<b>Comentario del revisor. (Iteración 1)</b> <a class="tocSkip"></a><br><br>

¡Buen esfuerzo construyendo el funnel mediante SQL, Cristian!

Quiero destacar que utilizaste varias CTEs para organizar las etapas del recorrido del usuario y que seleccionaste identificadores únicos mediante <code>DISTINCT id_usuario</code>.

Esta estructura demuestra una buena comprensión de cómo descomponer una consulta compleja en partes más manejables.

Sin embargo, encontré una oportunidad de mejora en el cálculo de las conversiones.

Actualmente estás dividiendo los usuarios de cada etapa entre el total de usuarios de <code>first_visit</code>.

Este cálculo permite obtener una conversión general respecto al inicio del funnel, pero no representa la conversión entre etapas consecutivas.

Para identificar correctamente dónde ocurre la mayor pérdida de usuarios, necesitamos comparar cada paso con el anterior.

Por ejemplo, para analizar la transición de <code>begin_checkout</code> a <code>add_payment_info</code>, el denominador debería corresponder a los usuarios que alcanzaron <code>begin_checkout</code>, considerando idealmente el avance secuencial de los usuarios.

También te recomiendo revisar la división numérica en SQL para evitar que una división entre enteros elimine los decimales antes del redondeo.

<b>Recomendación:</b> Conserva tu estructura con CTEs, pero agrega las tasas de conversión entre etapas consecutivas y utiliza esos porcentajes para confirmar dónde se presenta el mayor abandono.

No necesitas rehacer todo el análisis: la estructura que desarrollaste constituye un buen punto de partida.

Este ajuste permitirá que tus recomendaciones de negocio estén mejor fundamentadas.

</div>

---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users`
- `user_activity`

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [99]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''

users = pd.read_sql(query_users, con=engine)
users.head(3)


,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [100]:
# Explorar tabla user_activity
# =========================

query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(4)


,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1
3,user_0,2025-02-26,28,0


In [101]:
# Tatal usuarios registrados por cohortes
query_users = '''
SELECT TO_CHAR(DATE_TRUNC('month', CAST(fecha_registro AS DATE)), 'YYYY-MM') AS cohorte_mes,
       COUNT(DISTINCT id_usuario) AS total_usuarios
FROM users
GROUP BY cohorte_mes
ORDER BY cohorte_mes ASC;
'''
total_users = pd.read_sql(query_users, con=engine)
total_users.head()

,cohorte_mes,total_usuarios
0,2025-01,1627
1,2025-02,1444
2,2025-03,1636
3,2025-04,1606
4,2025-05,1687


In [102]:
# Retención por cohortes
query_cohort_retention_final = '''
SELECT TO_CHAR(DATE_TRUNC('month', CAST(u.fecha_registro AS DATE)), 'YYYY-MM') AS Cohorte_mes,
COUNT(DISTINCT CASE WHEN ua.dias_despues_registro >= 7 AND ua.activo = 1 THEN ua.id_usuario ELSE NULL END) AS retenido_w1,
COUNT(DISTINCT CASE WHEN ua.dias_despues_registro >= 14 AND ua.activo = 1 THEN ua.id_usuario ELSE NULL END) AS retenido_w2,
COUNT(DISTINCT CASE WHEN ua.dias_despues_registro >= 21 AND ua.activo = 1 THEN ua.id_usuario ELSE NULL END) AS retenido_w3
FROM users u
LEFT JOIN user_activity ua
    ON u.id_usuario = ua.id_usuario
GROUP BY cohorte_mes
'''

cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte_mes,retenido_w1,retenido_w2,retenido_w3
0,2025-01,1381,1253,1027
1,2025-02,1255,1154,940
2,2025-03,1428,1306,1060
3,2025-04,1394,1261,1022
4,2025-05,1446,1321,1088


In [103]:
# Porcentaje de retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH total_usuarios AS(
SELECT COUNT(DISTINCT(id_usuario)) AS total_usuarios
FROM users
),
cte_retencion AS(
SELECT TO_CHAR(DATE_TRUNC('month', CAST(u.fecha_registro AS DATE)), 'YYYY-MM') AS Cohorte_mes,
COUNT(DISTINCT u.id_usuario) AS total_usuarios,
COUNT(DISTINCT CASE WHEN ua.dias_despues_registro >= 7 AND ua.activo = 1 THEN ua.id_usuario ELSE NULL END) AS retenido_w1,
COUNT(DISTINCT CASE WHEN ua.dias_despues_registro >= 14 AND ua.activo = 1 THEN ua.id_usuario ELSE NULL END) AS retenido_w2,
COUNT(DISTINCT CASE WHEN ua.dias_despues_registro >= 21 AND ua.activo = 1 THEN ua.id_usuario ELSE NULL END) AS retenido_w3,
COUNT(DISTINCT CASE WHEN ua.dias_despues_registro >= 28 AND ua.activo = 1 THEN ua.id_usuario ELSE NULL END) AS retenido_w4
FROM users u
LEFT JOIN user_activity ua
    ON u.id_usuario = ua.id_usuario
GROUP BY cohorte_mes
)
SELECT
Cohorte_mes,
ROUND(retenido_w1::numeric / total_usuarios * 100, 2) AS semana_1,
ROUND(retenido_w2::numeric / total_usuarios * 100, 2) AS semana_2,
ROUND(retenido_w3::numeric / total_usuarios * 100, 2) AS semana_3
FROM cte_retencion;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte_mes,semana_1,semana_2,semana_3
0,2025-01,84.88,77.01,63.12
1,2025-02,86.91,79.92,65.10
2,2025-03,87.29,79.83,64.79
3,2025-04,86.80,78.52,63.64
4,2025-05,85.71,78.30,64.49


La cohorte que tuvo la menor tasa de retención en semana_3 es `2025-01`. Revisar este impacto para implementar la estrategía comercial.

<div class="alert alert-block alert-success">
<b>Comentario del revisor. (Iteración 1)</b> <a class="tocSkip"></a><br><br>

¡Muy buen trabajo con el análisis de retención por cohortes, Cristian!

Esta es una de las partes más interesantes de tu entrega.

Utilizaste <code>DATE_TRUNC('month', CAST(fecha_registro AS DATE))</code> para construir las cohortes según el mes de registro de los usuarios.

También empleaste <code>COUNT(DISTINCT ...)</code> para calcular los usuarios retenidos, evitando contar varias veces a una misma persona.

Otro aspecto positivo es que trabajaste con las condiciones de actividad a partir de los días 7, 14 y 21 posteriores al registro, lo que permite obtener una lógica acumulada equivalente a los umbrales semanales solicitados.

Finalmente, calculaste los porcentajes de retención utilizando como denominador el total de usuarios de cada cohorte.

Esto es importante porque permite comparar grupos de distintos tamaños.

Tu interpretación también identifica que la cohorte de enero de 2025 presenta el menor porcentaje de retención en la tercera semana.

Como recomendación opcional, podrías complementar esa conclusión con una breve hipótesis de negocio sobre los factores que podrían explicar ese comportamiento.

En general, es una sección bien desarrollada y alineada con los objetivos del proyecto.

¡Felicitaciones por este trabajo!

</div>

---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado**
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** Afirma que no existe una diferencia real en la duración de sesiones de usuarios entre ambos grupos.
   - **H₁ (Hipótesis alternativa):** Afirma que existe una diferencia real en la duración de sesiones de usuarios entre ambos grupos.
   
**Test estadístico:** Prueba t-test.    
**Nivel de significancia alpha:** Umbral que nos permite determinar si los datos son confiables. Umbral de significancia alpha = < 0.05.

In [104]:
# Importar y explorar dataset
experiment = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')
experiment.head()

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [105]:
# Revisión tipos de datos
experiment.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_usuario       10000 non-null  object 
 1   variante         10000 non-null  object 
 2   convirtio        10000 non-null  int64  
 3   dispositivo      10000 non-null  object 
 4   pais             10000 non-null  object 
 5   duracion_sesion  10000 non-null  float64
 6   timestamp        10000 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 547.0+ KB


In [106]:
# División de grupos
grupo_tratamiento = experiment[experiment['variante'] == 'tratamiento']['duracion_sesion']
grupo_control = experiment[experiment['variante'] == 'control']['duracion_sesion']

# Verificamos el tamaño de cada grupo
len(grupo_tratamiento), len(grupo_control)

(5035, 4965)

**NOTA:** la columna timestap tiene un formato de dato inadecuado, sin embargo, para el siguiente analisis no tiene relevancia.

In [107]:
# Aplicación prueba t-test
from scipy.stats import ttest_ind

t_stat, p_value = ttest_ind(grupo_tratamiento, grupo_control, equal_var=False)
print(f"Estadístico: {t_stat}")
print(f"Valor p: {p_value}")

Estadístico: -1.445362141787994
Valor p: 0.1483875510051254


**No se rechaza hipotesis nula**

El resultado de la prueba t-test indica que no existe evidencia suficiente para afirmar que hay una diferencia real entre ambos grupos. Esto teniendo en cuenta que el valor p supera el umbral de significancia alpha. Como interpretación puntual, no podemos determinar si alguna de las dos versiones genera mayor eficiencia en el proceso de conversión.

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** Afirma que no existe una diferencia real entre los usuarios que compraron en cada grupo.
   - **H₁ (Hipótesis alternativa):** Afirma que existe una diferencia real entre los usuarios que compraron en cada grupo.
   
**Test estadístico:** Prueba de proporciones.    
**Nivel de significancia alpha:** Umbral que nos permite determinar si los datos son confiables. Umbral de significancia alpha = < 0.05.

In [108]:
# Conversión por grupo y total para proporciones

conversion = experiment[experiment['convirtio'] == 1]['variante'].value_counts()
print('Usuarios convertidos por página:',conversion)
print() # salto de linea
total_usuarios = experiment['variante'].value_counts()
print('Total usuarios por grupo:',total_usuarios)

Usuarios convertidos por página: variante
tratamiento    820
control        779
Name: count, dtype: int64

Total usuarios por grupo: variante
tratamiento    5035
control        4965
Name: count, dtype: int64


In [109]:
# Aplicación prueba de proporciones
from statsmodels.stats.proportion import proportions_ztest

conversiones = [conversion['tratamiento'], conversion['control']]
totales = [total_usuarios['tratamiento'], total_usuarios['control']]

z_stat, p_value = proportions_ztest(conversiones, totales)

print(f"Estadístico : {z_stat}")
print(f"Valor p: {p_value}")

Estadístico : 0.8132782986429474
Valor p: 0.41605851639119995


**No se rechaza hipotesis nula**

La prueba de proporciones aplicada indica que no existe evidencia suficiente para afirmar que hay una diferencia real entre ambos grupos. Nuevamente el valor de significancia alpha se encuentra por fuera del umbral. En conclusión, no podemos determinar si alguna de las versiones causa un mayor impacto en el volumen de ventas.

<div class="alert alert-block alert-warning">
<b>Comentario del revisor. (Iteración 1)</b> <a class="tocSkip"></a><br><br>

Cristian, me parece positivo que hayas explorado una prueba estadística adicional para comparar la duración de las sesiones entre los grupos de control y tratamiento.

La utilización de <code>ttest_ind()</code> con <code>equal_var=False</code> permite evaluar diferencias entre los promedios de duración de sesión de ambos grupos.

Sin embargo, hay una distinción conceptual importante.

Esta prueba evalúa la duración de las sesiones, mientras que el objetivo principal del experimento solicitado consiste en comparar las tasas de conversión de compra.

Por lo tanto, el resultado de esta prueba no permite concluir directamente que una variante tiene mayor o menor eficiencia en términos de compras.

<b>Recomendación:</b> Puedes conservar este análisis como una exploración complementaria, pero procura diferenciar su interpretación de la prueba de proporciones que realizas posteriormente.

Así quedará claro que cada prueba responde una pregunta estadística diferente.

Es positivo que quieras explorar más allá de lo obligatorio. La clave está en conectar cada prueba con la variable de negocio que realmente estás evaluando.

</div>

---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión.

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

https://drive.google.com/file/d/1BcgvPXgDmx4T-MUbk0rZ0DddqBvVwvDN/view?usp=sharing
Adicionalmente, dejo el archivo en la carpeta en el presente notebook en caso de que hayan inconvenientes con el link

<div class="alert alert-block alert-success">
<b>Comentario del revisor. (Iteración 1)</b> <a class="tocSkip"></a><br><br>

Cristian, también quiero reconocer el esfuerzo dedicado a comunicar los resultados mediante una herramienta de visualización de datos.

El dashboard se considera, en términos generales, acorde con el nivel esperado para este proyecto.

Recuerda que el objetivo principal de un dashboard no es desarrollar una presentación visual excesivamente compleja, sino ofrecer información clara y útil para comprender el desempeño del negocio.

Como recomendación de consistencia, una vez realizada la limpieza pendiente en Python, asegúrate de actualizar las fuentes de información y validar que los KPIs financieros del dashboard coincidan con los resultados definitivos del notebook.

De esta manera, tu análisis técnico y su presentación visual comunicarán los mismos resultados.

No es necesario realizar modificaciones estéticas adicionales para superar esta etapa.

¡Buen esfuerzo con el desarrollo del proyecto!

</div>